In [1]:
# 1. Import libraries and helper functions

import pandas as pd
from pathlib import Path
import os

def normalize_text(series):
    return series.astype(str).str.strip().str.title()

In [2]:
# 2. Path configuration

BASE_DIR = Path(os.environ.get("MASKING_DATA_PATH", "../data"))
RAW_DIR = BASE_DIR / "raw" / "us"
PROCESSED_DIR = BASE_DIR / "processed" / "us"

In [3]:
# 3. Load dataset

df = pd.read_excel(RAW_DIR / "products_raw.xlsx").dropna(how='all')

In [4]:
#display(df.head())

In [5]:
# 4. Filter dataset columns

filtered_columns = [
    'ITEM_ID', 'PRODUCT_NAME', 'PRODUCT_LINE', 'PROD_GROUP', 'BRAND', 'DEPARTMENT', 'CATEGORY', 'SUB_CATEGORY'
]

df = df[filtered_columns].copy()

In [6]:
# 5. Rename columns

df.columns = df.columns.str.lower()
df = df.rename(columns={"prod_group": "product_group"})

In [7]:
# 6. Validate before changing anything

assert df['item_id'].duplicated().sum() == 0, "item_id duplicated!"

dupes_count = df['product_name'].duplicated().sum()
print(f"{dupes_count} duplicate product_name records found - resolved in step 7")

648 duplicate product_name records found - resolved in step 7


In [8]:
# 7. Handle duplicate product cadastration

# Multiple ITEM_IDs can share the same PRODUCT_NAME due to duplicate
# registration in the source system (e.g., kits, bundles, re-entries).
# Keep the oldest ITEM_ID per name (lowest ID = first registration) and
# build a map from every removed ITEM_ID to the canonical one that
# survives, so downstream tables (sales, returns) can reassign any
# transaction instead of losing revenue history.

df = df.sort_values('item_id')

canonical_ids = (
    df.drop_duplicates(subset='product_name', keep='first')
    .set_index('product_name')['item_id']
)

item_id_map = {}
for name, group in df.groupby('product_name'):
    if len(group) > 1:
        canonical = canonical_ids[name]
        for old_id in group['item_id']:
            if old_id != canonical:
                item_id_map[old_id] = canonical

print(f"{len(item_id_map)} duplicate ITEM_IDs identified for {df['product_name'].duplicated().sum()} duplicated PRODUCT_NAME records")

648 duplicate ITEM_IDs identified for 648 duplicated PRODUCT_NAME records


In [9]:
# 8. Remove redundant records from the dimension

df = df[~df['item_id'].isin(item_id_map.keys())].copy()

assert df['product_name'].dropna().duplicated().sum() == 0, "product_name duplicated!"
assert df['item_id'].duplicated().sum() == 0, "item_id duplicated!"

In [10]:
# 9. Convert data types

df['item_id'] = df['item_id'].astype(str)
df['product_name'] = normalize_text(df['product_name'])
df['product_line'] = normalize_text(df['product_line'])
df['product_group'] = normalize_text(df['product_group']).astype('category')
df['brand'] = normalize_text(df['brand'])
df['department'] = normalize_text(df['department']).astype('category')
df['category'] = normalize_text(df['category']).astype('category')
df['sub_category'] = normalize_text(df['sub_category']).astype('category')

In [11]:
#df.info()

In [12]:
# 10. Final checks

assert df['item_id'].duplicated().sum() == 0, "item_id duplicated after cleaning!"
assert df['product_name'].dropna().duplicated().sum() == 0, "product_name still duplicated!"

In [13]:
# 11. Export cleaned dataset and the ITEM_ID map for downstream tables

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / "dim_products.csv", index=False)

item_id_map_df = pd.DataFrame(
    list(item_id_map.items()), columns=['old_item_id', 'canonical_item_id']
)
item_id_map_df.to_csv(PROCESSED_DIR / "item_id_map.csv", index=False)

print(f"Success! {len(df)} products exported, {len(item_id_map)} duplicate ITEM_IDs mapped to their canonical record.")

Success! 10821 products exported, 648 duplicate ITEM_IDs mapped to their canonical record.
